# Stage 3 — Experimental Investigation

**COMP5329 / Deep Learning — University of Sydney, Semester 1 2026**

Three controlled experiments on the corrected QANet implementation.

| Section | Content |
|---------|-------------------|
| 0 | Environment setup |
| 1 | Shared configuration + helper functions |
| 2 | **Baseline** — Adam, lr=1e-3, dropout=0.1 (shared by all experiments) |
| 3 | **Experiment 1** — Optimizer: SGD / Adam / SGD+Momentum |
| 4 | **Experiment 2** — Learning rate: 1e-2 / 1e-3 / 1e-4 |
| 5 | **Experiment 3** — Dropout: 0.1 / 0.3 / 0.5 |
| 6 | Results summary and plots |

> **How to run independently**: Section 2 (Baseline) must be run once and its result is saved to `_log/baseline.json`. After that, Sections 3, 4, and 5 can each be run independently in any order — they load the baseline from the file rather than re-training it.

---
## Section 0 — Environment Setup

Run once per Colab session.

In [5]:
#from google.colab import drive
#drive.mount('/content/drive')

#PROJECT_ROOT = "/content/drive/MyDrive/COMP5329_Assignment1"
PROJECT_ROOT = "."

In [6]:
import sys, os

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

os.chdir(PROJECT_ROOT)
print("Working directory:", os.getcwd())

Working directory: /Users/xyx/Documents/Assignment1_2026


---
## Section 1 — Shared Configuration

**Must run this section before any experiment.**

Defines the shared `BASE` config and two helper functions:
- `run()` — trains one configuration and stores the result in `all_results`
- `load_baseline()` — loads the pre-trained baseline result from `_log/baseline.json`

In [7]:
import json, os
import matplotlib.pyplot as plt
from TrainTools.train import train

BASELINE_PATH = "_log/baseline.json"

# ── Shared base configuration ─────────────────────────────────────────────────
# Each experiment overrides ONLY the variable under investigation.
BASE = dict(
    # data paths
    train_npz       = "_data/train.npz",
    dev_npz         = "_data/dev.npz",
    word_emb_json   = "_data/word_emb.json",
    char_emb_json   = "_data/char_emb.json",
    train_eval_json = "_data/train_eval.json",
    dev_eval_json   = "_data/dev_eval.json",
    log_dir         = "_log",

    # training loop
    num_steps        = 3000,
    batch_size       = 8,
    checkpoint       = 500,
    val_num_batches  = 100,
    test_num_batches = 100,
    seed             = 42,
    grad_clip        = 5.0,
    early_stop       = 20,

    # fixed architecture — never changed across experiments
    d_model      = 96,
    num_heads    = 8,
    glove_dim    = 300,
    char_dim     = 64,
    para_limit   = 400,
    ques_limit   = 50,
    char_limit   = 16,
    norm_name    = "layer_norm",
    activation   = "relu",
    init_name    = "kaiming",

    # default training components
    loss_name      = "qa_nll",
    optimizer_name = "adam",
    scheduler_name = "none",
    learning_rate  = 1e-3,
    beta1          = 0.8,
    beta2          = 0.999,
    eps            = 1e-7,
    weight_decay   = 3e-7,
    momentum       = 0.9,
    dropout        = 0.1,
    dropout_char   = 0.05,
)

# Storage for current session's results
all_results = {}


def run(label, **overrides):
    """Train one configuration and store the result in all_results."""
    save_dir = os.path.join("_model", label)
    cfg = {**BASE, "save_dir": save_dir, **overrides}
    print(f"\n{'='*60}")
    print(f"  Running : {label}")
    print(f"  Changed : { {k: v for k, v in overrides.items()} }")
    print(f"{'='*60}")
    r = train(**cfg)
    all_results[label] = {
        "best_f1": r["best_f1"],
        "best_em": r["best_em"],
        "history": r["history"],
    }
    print(f"\n  >>> {label}:  Best F1={r['best_f1']:.4f}   EM={r['best_em']:.4f}")
    return r


def load_baseline():
    """
    Load the pre-trained baseline result from disk into all_results.
    Raises a clear error if Section 2 has not been run yet.
    """
    if not os.path.exists(BASELINE_PATH):
        raise FileNotFoundError(
            f"Baseline file not found: {BASELINE_PATH}\n"
            "Please run Section 2 first to train and save the baseline."
        )
    with open(BASELINE_PATH, "r") as f:
        data = json.load(f)
    all_results["baseline"] = data
    print(f"Baseline loaded from {BASELINE_PATH}")
    print(f"  Best F1={data['best_f1']:.4f}   EM={data['best_em']:.4f}")


os.makedirs("_log", exist_ok=True)
print("Shared configuration loaded. Ready to run experiments.")

Shared configuration loaded. Ready to run experiments.


---
## Section 2 — Baseline Training

**Run this section once.** The result is saved to `_log/baseline.json` and reused by all three experiments.

| Parameter | Value |
|-----------|-------|
| optimizer | Adam |
| learning_rate | 1e-3 |
| scheduler | none (constant LR) |
| dropout | 0.1 |

> If `_log/baseline.json` already exists (from a previous session), you can skip this section and go directly to any experiment.

In [8]:
# Check whether the baseline has already been trained
if os.path.exists(BASELINE_PATH):
    print(f"Baseline already exists at {BASELINE_PATH}")
    print("Skip to the experiment section you want to run.")
    print("Or delete the file and re-run this cell to retrain.")
else:
    print("Baseline not found. Run the next cell to train it.")

Baseline not found. Run the next cell to train it.


In [9]:
# Train the baseline (Adam, lr=1e-3, dropout=0.1)
# Skip this cell if baseline.json already exists.
run("baseline",
    optimizer_name = "adam",
    scheduler_name = "none",
    learning_rate  = 1e-3,
    dropout        = 0.1,
    dropout_char   = 0.05,
)

# Save baseline result to disk so it can be reused across sessions
with open(BASELINE_PATH, "w") as f:
    json.dump(all_results["baseline"], f, indent=2)
print(f"\nBaseline saved to {BASELINE_PATH}")


  Running : baseline
  Changed : {'optimizer_name': 'adam', 'scheduler_name': 'none', 'learning_rate': 0.001, 'dropout': 0.1, 'dropout_char': 0.05}


100%|█████████████████████████████████████████| 500/500 [57:39<00:00,  6.92s/it]


STEP      500  loss 6.121165



100%|█████████████████████████████████████████| 100/100 [03:47<00:00,  2.28s/it]


VALID(train) loss 3.701302  F1 12.744578  EM 4.750000



...

STEP     3000  loss 3.135844



100%|█████████████████████████████████████████| 100/100 [01:32<00:00,  1.09it/s]


VALID(train) loss 2.877493  F1 23.032784  EM 15.125000



100%|█████████████████████████████████████████| 100/100 [01:33<00:00,  1.07it/s]


TEST        loss 3.363277  F1 20.698501  EM 13.125000

Learning rate: [0.001]
Training finished.  Best F1: 21.4244  Best EM: 14.6250

  >>> baseline:  Best F1=21.4244   EM=14.6250

Baseline saved to _log/baseline.json


---
## Section 3 — Experiment 1: Optimizer Comparison

**Can be run independently** after Section 2 has been completed at least once.

**Research Question**: How does optimizer choice affect convergence speed and final QA performance?

**Hypothesis**: Adam converges faster than SGD due to adaptive per-parameter learning rates. SGD+Momentum bridges the gap by accumulating gradient direction but lacking adaptivity.

**Controlled variables**: `lr=1e-3`, `scheduler=none`, `dropout=0.1`  
**Independent variable**: `optimizer_name`

| Run | optimizer | Expected behaviour |
|-----|-----------|--------------------|
| Baseline | `adam` | Loaded from file |
| Exp A | `sgd` | Slow convergence, loss plateaus early |
| Exp B | `sgd_momentum` | Faster than SGD, slower than Adam |

In [10]:
# Load baseline from file (no re-training)
load_baseline()
all_results["exp1_adam"] = all_results["baseline"]

Baseline loaded from _log/baseline.json
  Best F1=21.4244   EM=14.6250


In [11]:
# Experiment A — Vanilla SGD
run("exp1_sgd",
    optimizer_name = "sgd",
    scheduler_name = "none",
    learning_rate  = 1e-3,
)


  Running : exp1_sgd
  Changed : {'optimizer_name': 'sgd', 'scheduler_name': 'none', 'learning_rate': 0.001}


100%|█████████████████████████████████████████| 500/500 [50:35<00:00,  6.07s/it]


STEP      500  loss 25.457077



100%|█████████████████████████████████████████| 100/100 [01:33<00:00,  1.07it/s]


VALID(train) loss 7.706373  F1 6.441853  EM 0.250000



...

100%|█████████████████████████████████████████| 100/100 [01:32<00:00,  1.09it/s]


VALID(train) loss 4.711328  F1 6.372553  EM 0.250000



100%|█████████████████████████████████████████| 100/100 [01:31<00:00,  1.10it/s]


TEST        loss 4.768713  F1 6.520387  EM 0.125000

Learning rate: [0.001]
Training finished.  Best F1: 7.0608  Best EM: 0.2500

  >>> exp1_sgd:  Best F1=7.0608   EM=0.2500


{'best_f1': 7.060823291541337,
 'best_em': 0.25,
 'history': [{'step': 500,
   'train_loss': 25.457077322006224,
   'train_f1': 6.441852925137287,
   'train_em': 0.25,
   'dev_loss': 7.787957572937012,
   'dev_f1': 5.995791712940174,
   'dev_em': 0.0,
   'lr': 0.001},
  {'step': 1000,
   'train_loss': 9.299370150566102,
   'train_f1': 6.414242278068907,
   'train_em': 0.125,
   'dev_loss': 5.33413254737854,
   'dev_f1': 5.985143741518961,
   'dev_em': 0.125,
   'lr': 0.001},
  {'step': 1500,
   'train_loss': 6.458799111366272,
   'train_f1': 6.090180988886702,
   'train_em': 0.125,
   'dev_loss': 5.000053739547729,
   'dev_f1': 5.28557651715143,
   'dev_em': 0.0,
   'lr': 0.001},
  {'step': 2000,
   'train_loss': 5.645545964241028,
   'train_f1': 6.659270384680863,
   'train_em': 0.0,
   'dev_loss': 4.836415252685547,
   'dev_f1': 7.060823291541337,
   'dev_em': 0.125,
   'lr': 0.001},
  {'step': 2500,
   'train_loss': 5.3594283723831175,
   'train_f1': 7.149814134334721,
   'train_em'

In [ ]:
# Experiment B — SGD with Momentum
run("exp1_sgdm",
    optimizer_name = "sgd_momentum",
    scheduler_name = "none",
    learning_rate  = 1e-3,
    momentum       = 0.9,
)


  Running : exp1_sgdm
  Changed : {'optimizer_name': 'sgd_momentum', 'scheduler_name': 'none', 'learning_rate': 0.001, 'momentum': 0.9}


  3%|█▍                                        | 17/500 [01:01<28:45,  3.57s/it]

In [ ]:
# Results table
print("\n" + "="*50)
print("  Experiment 1: Optimizer Comparison")
print("="*50)
print(f"{'Config':<20} {'Best F1':>10} {'Best EM':>10}")
print("-" * 45)
for key, label in [("exp1_adam", "Adam (baseline)"),
                   ("exp1_sgd",  "SGD"),
                   ("exp1_sgdm", "SGD+Momentum")]:
    r = all_results[key]
    print(f"{label:<20} {r['best_f1']:>10.4f} {r['best_em']:>10.4f}")

# Loss curve
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
variants = [("exp1_adam", "Adam (baseline)", "tab:orange"),
            ("exp1_sgd",  "SGD",             "tab:blue"),
            ("exp1_sgdm", "SGD+Momentum",    "tab:green")]
for metric, ax, ylabel in [("train_loss", axes[0], "Training Loss"),
                            ("dev_f1",    axes[1], "Dev F1 (%)")]:
    for key, label, color in variants:
        if key not in all_results:
            continue
        h = all_results[key]["history"]
        ax.plot([e["step"] for e in h], [e[metric] for e in h],
                marker="o", label=label, color=color)
    ax.set_title(f"Exp 1: {ylabel} — Optimizer")
    ax.set_xlabel("Training Steps")
    ax.set_ylabel(ylabel)
    ax.legend()
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("_log/exp1_optimizer.png", dpi=150, bbox_inches="tight")
plt.show()
print("Plot saved to _log/exp1_optimizer.png")

---
## Section 4 — Experiment 2: Learning Rate Sensitivity

**Can be run independently** after Section 2 has been completed at least once.

**Research Question**: How sensitive is QANet training to the choice of initial learning rate?

**Hypothesis**: `lr=1e-3` achieves the best balance. `lr=1e-2` causes loss oscillation; `lr=1e-4` converges too slowly to reach meaningful F1 within 3000 steps.

**Controlled variables**: `optimizer=adam`, `scheduler=none`, `dropout=0.1`  
**Independent variable**: `learning_rate`

| Run | learning_rate | Expected behaviour |
|-----|--------------|--------------------|
| Exp A | `1e-2` | Loss oscillates or diverges |
| Baseline | `1e-3` | Stable descent — loaded from file |
| Exp B | `1e-4` | Very slow descent, low F1 at step 3000 |

In [ ]:
# Load baseline from file (no re-training)
load_baseline()
all_results["exp2_lr1e3"] = all_results["baseline"]

In [ ]:
# Experiment A — large learning rate
run("exp2_lr1e2",
    optimizer_name = "adam",
    scheduler_name = "none",
    learning_rate  = 1e-2,
)

In [ ]:
# Experiment B — small learning rate
run("exp2_lr1e4",
    optimizer_name = "adam",
    scheduler_name = "none",
    learning_rate  = 1e-4,
)

In [ ]:
# Results table
print("\n" + "="*50)
print("  Experiment 2: Learning Rate Sensitivity")
print("="*50)
print(f"{'Config':<22} {'Best F1':>10} {'Best EM':>10}")
print("-" * 47)
for key, label in [("exp2_lr1e2", "Adam lr=1e-2"),
                   ("exp2_lr1e3", "Adam lr=1e-3 (baseline)"),
                   ("exp2_lr1e4", "Adam lr=1e-4")]:
    r = all_results[key]
    print(f"{label:<22} {r['best_f1']:>10.4f} {r['best_em']:>10.4f}")

# Loss curve
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
variants = [("exp2_lr1e2", "lr=1e-2", "tab:red"),
            ("exp2_lr1e3", "lr=1e-3 (baseline)", "tab:orange"),
            ("exp2_lr1e4", "lr=1e-4", "tab:blue")]
for metric, ax, ylabel in [("train_loss", axes[0], "Training Loss"),
                            ("dev_f1",    axes[1], "Dev F1 (%)")]:
    for key, label, color in variants:
        if key not in all_results:
            continue
        h = all_results[key]["history"]
        ax.plot([e["step"] for e in h], [e[metric] for e in h],
                marker="o", label=label, color=color)
    ax.set_title(f"Exp 2: {ylabel} — Learning Rate")
    ax.set_xlabel("Training Steps")
    ax.set_ylabel(ylabel)
    ax.legend()
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("_log/exp2_learning_rate.png", dpi=150, bbox_inches="tight")
plt.show()
print("Plot saved to _log/exp2_learning_rate.png")

---
## Section 5 — Experiment 3: Dropout Regularization

**Can be run independently** after Section 2 has been completed at least once.

**Research Question**: Does increasing dropout rate improve generalization in QANet, and at what cost to training performance?

**Hypothesis**: Moderate dropout (`0.3`) may reduce overfitting and improve dev F1. Aggressive dropout (`0.5`) hurts training capacity too severely within 3000 steps.

**Controlled variables**: `optimizer=adam`, `lr=1e-3`, `scheduler=none`, `dropout_char=0.05`  
**Independent variable**: `dropout`

| Run | dropout | Expected behaviour |
|-----|---------|--------------------|
| Baseline | `0.1` | Slight overfitting — loaded from file |
| Exp A | `0.3` | Lower train F1, potentially higher dev F1 |
| Exp B | `0.5` | Both train and dev F1 drop significantly |

In [ ]:
# Load baseline from file (no re-training)
load_baseline()
all_results["exp3_drop01"] = all_results["baseline"]

In [ ]:
# Experiment A — moderate dropout
run("exp3_drop03",
    optimizer_name = "adam",
    scheduler_name = "none",
    learning_rate  = 1e-3,
    dropout        = 0.3,
    dropout_char   = 0.05,
)

In [ ]:
# Experiment B — aggressive dropout
run("exp3_drop05",
    optimizer_name = "adam",
    scheduler_name = "none",
    learning_rate  = 1e-3,
    dropout        = 0.5,
    dropout_char   = 0.05,
)

In [ ]:
# Results table
print("\n" + "="*50)
print("  Experiment 3: Dropout Regularization")
print("="*50)
print(f"{'Config':<24} {'Best F1':>10} {'Best EM':>10}")
print("-" * 48)
for key, label in [("exp3_drop01", "dropout=0.1 (baseline)"),
                   ("exp3_drop03", "dropout=0.3"),
                   ("exp3_drop05", "dropout=0.5")]:
    r = all_results[key]
    print(f"{label:<24} {r['best_f1']:>10.4f} {r['best_em']:>10.4f}")

# Loss curve + train vs dev F1 gap (overfitting analysis)
fig, axes = plt.subplots(1, 3, figsize=(18, 4))
variants = [("exp3_drop01", "dropout=0.1 (baseline)", "tab:blue"),
            ("exp3_drop03", "dropout=0.3",             "tab:orange"),
            ("exp3_drop05", "dropout=0.5",             "tab:red")]

for metric, ax, ylabel in [("train_loss", axes[0], "Training Loss"),
                            ("dev_f1",    axes[1], "Dev F1 (%)"),
                            ("train_f1",  axes[2], "Train F1 (%)")]:
    for key, label, color in variants:
        if key not in all_results:
            continue
        h = all_results[key]["history"]
        ax.plot([e["step"] for e in h], [e[metric] for e in h],
                marker="o", label=label, color=color)
    ax.set_title(f"Exp 3: {ylabel} — Dropout")
    ax.set_xlabel("Training Steps")
    ax.set_ylabel(ylabel)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("_log/exp3_dropout.png", dpi=150, bbox_inches="tight")
plt.show()
print("Plot saved to _log/exp3_dropout.png")

---
## Section 6 — Overall Summary

Aggregates results from all experiments into one table and one combined figure.  
Run this section only after all experiments are complete.

In [ ]:
# Save all results to JSON
summary = {k: {"best_f1": v["best_f1"], "best_em": v["best_em"]}
           for k, v in all_results.items()}
with open("_log/stage3_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

with open("_log/stage3_history.json", "w") as f:
    json.dump({k: v["history"] for k, v in all_results.items()}, f, indent=2)

print("Saved: _log/stage3_summary.json")
print("Saved: _log/stage3_history.json")

In [ ]:
# Combined 2x3 figure: training loss + dev F1 for all three experiments
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

plot_specs = [
    ("Exp 1: Training Loss — Optimizer",
     [("exp1_adam", "Adam (baseline)", "tab:orange"),
      ("exp1_sgd",  "SGD",             "tab:blue"),
      ("exp1_sgdm", "SGD+Momentum",    "tab:green")],
     "train_loss"),
    ("Exp 2: Training Loss — Learning Rate",
     [("exp2_lr1e2", "lr=1e-2",             "tab:red"),
      ("exp2_lr1e3", "lr=1e-3 (baseline)",  "tab:orange"),
      ("exp2_lr1e4", "lr=1e-4",             "tab:blue")],
     "train_loss"),
    ("Exp 3: Training Loss — Dropout",
     [("exp3_drop01", "dropout=0.1 (baseline)", "tab:blue"),
      ("exp3_drop03", "dropout=0.3",             "tab:orange"),
      ("exp3_drop05", "dropout=0.5",             "tab:red")],
     "train_loss"),
    ("Exp 1: Dev F1 — Optimizer",
     [("exp1_adam", "Adam (baseline)", "tab:orange"),
      ("exp1_sgd",  "SGD",             "tab:blue"),
      ("exp1_sgdm", "SGD+Momentum",    "tab:green")],
     "dev_f1"),
    ("Exp 2: Dev F1 — Learning Rate",
     [("exp2_lr1e2", "lr=1e-2",             "tab:red"),
      ("exp2_lr1e3", "lr=1e-3 (baseline)",  "tab:orange"),
      ("exp2_lr1e4", "lr=1e-4",             "tab:blue")],
     "dev_f1"),
    ("Exp 3: Dev F1 — Dropout",
     [("exp3_drop01", "dropout=0.1 (baseline)", "tab:blue"),
      ("exp3_drop03", "dropout=0.3",             "tab:orange"),
      ("exp3_drop05", "dropout=0.5",             "tab:red")],
     "dev_f1"),
]

for ax, (title, variants, metric) in zip(axes.flat, plot_specs):
    for key, label, color in variants:
        if key not in all_results:
            continue
        h = all_results[key]["history"]
        ax.plot([e["step"] for e in h], [e[metric] for e in h],
                marker="o", label=label, color=color)
    ax.set_title(title, fontsize=11)
    ax.set_xlabel("Training Steps")
    ax.set_ylabel("Training Loss" if metric == "train_loss" else "Dev F1 (%)")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.suptitle("QANet Stage 3 — All Experimental Results", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("_log/stage3_results.png", dpi=150, bbox_inches="tight")
plt.show()
print("Combined plot saved to _log/stage3_results.png")

In [ ]:
# Final summary table
print("\n" + "="*55)
print("  STAGE 3 COMPLETE SUMMARY")
print("="*55)
print(f"{'Run':<28} {'Best F1':>10} {'Best EM':>10}")
print("-" * 52)
display_order = [
    ("exp1_adam",   "Exp1  Adam (baseline)"),
    ("exp1_sgd",    "Exp1  SGD"),
    ("exp1_sgdm",   "Exp1  SGD+Momentum"),
    ("exp2_lr1e2",  "Exp2  lr=1e-2"),
    ("exp2_lr1e3",  "Exp2  lr=1e-3 (baseline)"),
    ("exp2_lr1e4",  "Exp2  lr=1e-4"),
    ("exp3_drop01", "Exp3  dropout=0.1 (baseline)"),
    ("exp3_drop03", "Exp3  dropout=0.3"),
    ("exp3_drop05", "Exp3  dropout=0.5"),
]
for key, label in display_order:
    if key in all_results:
        r = all_results[key]
        print(f"{label:<28} {r['best_f1']:>10.4f} {r['best_em']:>10.4f}")
    else:
        print(f"{label:<28} {'(not run yet)':>22}")